# Exploratory Data Analysis

Chicago Traffic Crash Analytics — Checkpoint 3

Research question: What factors predict whether a Chicago traffic crash
results in injury, and do time-of-day and weather effects persist after
controlling for road and crash characteristics?

This notebook loads the full 2023-2024 pull (see
`scripts/fetch_full_data.py`, data cached in `data/raw/*.parquet`), profiles
the two primary datasets, documents anomalies, and plots distributions and
relationships relevant to the research question.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (8, 5)

crashes = pd.read_parquet("../data/raw/crashes.parquet")
people = pd.read_parquet("../data/raw/people.parquet")


## 1. Dataset Profiles

### Traffic Crashes - Crashes


In [ ]:
crashes.shape

In [ ]:
crashes.dtypes

In [ ]:
crashes.isnull().sum().sort_values(ascending=False)

In [ ]:
crashes.describe(include='all').T

**What stands out:** _(fill in after reviewing the output above — note any
columns that are entirely one Socrata string dtype that should be numeric,
columns that are almost entirely null, categorical columns with far more
levels than expected, etc.)_


### Traffic Crashes - People

In [ ]:
people.shape

In [ ]:
people.dtypes

In [ ]:
people.isnull().sum().sort_values(ascending=False)

In [ ]:
people.describe(include='all').T

**What stands out:** _(fill in after reviewing the output above)_


## 2. Anomalies

_(Work through this section systematically: duplicates, missing values,
inconsistent formatting, wrong dtypes, invalid values, mixed units. For each
one found, note what it is, how many rows it affects, and what we did.)_


In [ ]:
# Duplicate crash records?
crashes["crash_record_id"].duplicated().sum()


In [ ]:
# Socrata returns everything as strings -- check numeric-looking columns
numeric_like = ["posted_speed_limit", "injuries_total", "injuries_fatal", "lane_cnt"]
crashes[numeric_like].apply(lambda c: pd.to_numeric(c, errors="coerce")).describe()


In [ ]:
# How many rows have an invalid/placeholder posted speed limit (e.g. 0 or absurdly high)?
spd = pd.to_numeric(crashes["posted_speed_limit"], errors="coerce")
((spd == 0) | (spd > 70)).sum()


In [ ]:
# Weather/lighting/surface "UNKNOWN" rates
for col in ["weather_condition", "lighting_condition", "roadway_surface_cond", "road_defect"]:
    print(col, (crashes[col] == "UNKNOWN").mean().round(3))


In [ ]:
# People: invalid ages
age = pd.to_numeric(people["age"], errors="coerce")
((age < 0) | (age > 100)).sum(), age.isnull().sum()


**Anomaly notes:** _(fill in a table here: anomaly | rows affected | what we did)_


## 3. Distributions

One plot per feature that matters for the research question: injury outcome,
weather, lighting, road surface, posted speed limit, crash hour.


In [ ]:
crashes["most_severe_injury"].value_counts().plot(kind="barh")
plt.xlabel("Number of crashes")
plt.ylabel("Most severe injury")
plt.title("Distribution of crash injury severity")
plt.tight_layout()
plt.show()


_Shape/description:_ ...

In [ ]:
pd.to_numeric(crashes["crash_hour"], errors="coerce").plot(kind="hist", bins=24)
plt.xlabel("Hour of day")
plt.title("Distribution of crash hour")
plt.tight_layout()
plt.show()


_Shape/description:_ ...

In [ ]:
crashes["weather_condition"].value_counts().plot(kind="barh")
plt.xlabel("Number of crashes")
plt.title("Distribution of weather condition at crash time")
plt.tight_layout()
plt.show()


_Shape/description:_ ...

In [ ]:
spd = pd.to_numeric(crashes["posted_speed_limit"], errors="coerce")
spd[(spd > 0) & (spd <= 70)].plot(kind="hist", bins=20)
plt.xlabel("Posted speed limit (mph)")
plt.title("Distribution of posted speed limit (invalid values excluded)")
plt.tight_layout()
plt.show()


_Shape/description:_ ...

## 4. Relationships

At least 3 plots relating two+ features, chosen with the research question
in mind.


### Relationship 1: Injury rate by hour of day

In [ ]:
crashes["injured"] = crashes["injuries_total"].astype(float).fillna(0) > 0
crashes["crash_hour_num"] = pd.to_numeric(crashes["crash_hour"], errors="coerce")
by_hour = crashes.groupby("crash_hour_num")["injured"].mean()
by_hour.plot(kind="line", marker="o")
plt.xlabel("Hour of day")
plt.ylabel("Share of crashes with injury")
plt.title("Injury rate by hour of day")
plt.tight_layout()
plt.show()


**So what?** ...

**What next?** ...

### Relationship 2: Injury rate by weather condition

In [ ]:
by_weather = crashes.groupby("weather_condition")["injured"].mean().sort_values()
by_weather.plot(kind="barh")
plt.xlabel("Share of crashes with injury")
plt.title("Injury rate by weather condition")
plt.tight_layout()
plt.show()


**So what?** ...

**What next?** ...

### Relationship 3: Injury rate by lighting condition x weather (subgroup comparison)

In [ ]:
pivot = crashes.pivot_table(
    index="lighting_condition", columns="weather_condition",
    values="injured", aggfunc="mean"
)
plt.imshow(pivot, aspect="auto", cmap="viridis")
plt.colorbar(label="Share of crashes with injury")
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=90)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.title("Injury rate by lighting x weather condition")
plt.tight_layout()
plt.show()


**So what?** ...

**What next?** ...

## 5. Research questions, revisited

_(Restate the research question here, noting anything the data has nudged.)_
